In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import poisson
from scipy.stats import norm
import matplotlib.pyplot as plt
import os
import matplotlib as mpl


# Load data 

In [ ]:
q = [0.1,0.5,0.9]
path='NegBinomValidation/'
chip=[pd.read_csv(f'{path}/chip_0.1.csv'),pd.read_csv(f'{path}/chip_0.5.csv'),pd.read_csv(f'{path}/chip_0.9.csv')]
Cov=[np.load(f'{path}/Cov_0.1.npy'),np.load(f'{path}/Cov_0.5.npy'),np.load(f'{path}/Cov_0.9.npy')]



In [ ]:
q_grid=np.arange(0.05,1, 0.05)
rho_grid=[0.3,0.35,0.4,0.45,0.5,0.55,0.6,0.65,0.7,0.75,0.8,0.85,0.9,0.95,1.0]

In [ ]:
KS_rho=np.load(f'{path}KS_rho.npy')
KS_rho_c=np.load(f'{path}KS_rho_c.npy')
KS_q=np.load(f'{path}KS_q.npy')
KS_q_c=np.load(f'{path}KS_q_c.npy')
KS_q_c_un=np.load(f'{path}KS_q_c_un.npy')
KS_q_s_un=np.load(f'{path}KS_q_s_un.npy')
KS_q_s=np.load(f'{path}KS_q_s.npy')

In [ ]:
KS_rho.shape

# Plot

In [ ]:
tex_fonts = {
    # Use LaTeX to write all text
    "text.usetex": True,
    "font.family": "serif",
    # Use 10pt font in plots, to match 10pt font in document
    "axes.labelsize": 10,
    "font.size": 10,
    # Make the legend/label fonts a little smaller
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8
}
tickParams = {
    "xtick.top":True,
    "xtick.bottom":True,
    "xtick.direction": "in",
    "ytick.left":True,
    "ytick.right":True,
    "ytick.direction": "in",
}
spinesParams = {
    'axes.spines.right' : True,
    'axes.spines.top' : True
}
plt.rcParams.update(tex_fonts)
plt.rcParams.update(tickParams)
inchPerCm = 0.393701
goldenRatio = (1+np.sqrt(5))/2
figWidthCm = 17.9
figHeightCm = figWidthCm/1
#figHeightCm = figWidthCm*1.2

figWidthInches = figWidthCm*inchPerCm
figHeightInches = figHeightCm*inchPerCm


lw=1
marker_size=10
lw2=1

colors=sns.color_palette("colorblind")
cmap_KL = mpl.colormaps['OrRd']

fig_label_x=-0.2
fig_label_y=1

In [ ]:
height=35
x = np.linspace(0, 1, 10000)


fig, ax = plt.subplots(3,2, figsize=(figWidthInches, figHeightInches))
#----------------------------q for all rows--------------------------------------------


for i in range(0,3):
    
    ax[i,0].plot(q[i] * np.ones(height),range(height), c='black',linewidth=lw2,linestyle='--') # target
    ax[i,0].plot(x, norm.pdf(x, q[i], np.sqrt(Cov[i][3,3])), c='black',linewidth=lw,label='sampling',linestyle='-')
    sns.kdeplot(data=chip[i]['q_chip_like'],c=colors[0],clip=(0,1),linewidth=lw,label='MLE with eq. (9)',linestyle='-',ax=ax[i,0])
    sns.kdeplot(data=chip[i]['q_chip'],c=colors[1],clip=(0,1),linewidth=lw,label=r'CE from eq. (11)',linestyle='-',ax=ax[i,0])  
    sns.kdeplot(data=chip[i]['q_chip_uncor'],c=colors[4],clip=(0,1),linewidth=lw,label=r'CE from eq. (11) with $\hat{q}_{|\tilde{0}}=1$',linestyle='-',ax=ax[i,0])
    
    ax[i,0].set_xlabel('')
    ax[i,0].set_ylabel('')


    ax[i,0].set_xlim([0,1])
    ax[i,0].set_ylim([0,35])
   



ax[2,0].set_xlabel('q')
ax[1,0].set_ylabel('pdf')

ax[0,0].text(fig_label_x, fig_label_y, '(a)', transform=ax[0,0].transAxes,
            size=11)

ax[0,0].legend(frameon=False,fancybox=True,handlelength=1.5)


#--------------------------------------KL divergences chip----------------------------------

c=ax[0,1].pcolormesh(q_grid,rho_grid,KS_q_c_un,shading='gouraud',vmin=0, vmax=10,cmap=cmap_KL)
cbar=fig.colorbar(c,ax=ax[0,1],ticks=np.arange(0,11,2),pad=0.01,label=r'KL-divergence-CE  with $\hat{q}_{|\tilde{0}}=1$')


c=ax[1,1].pcolormesh(q_grid,rho_grid,KS_q_c,shading='gouraud',vmin=0, vmax=0.5,cmap=cmap_KL)
cbar=fig.colorbar(c,ax=ax[1,1],ticks=np.arange(0,0.6,0.1),pad=0.01,label='KL-divergence-CE')



c=ax[2,1].pcolormesh(q_grid,rho_grid,KS_q,shading='gouraud',vmin=0,vmax=0.1,cmap=cmap_KL)
cbar=fig.colorbar(c,ax=ax[2,1],pad=0.01,label='KL-divergence-MLE')






#---------------------------modfy all axis of second column---------------------------------------------

ax[0,1].text(fig_label_x, fig_label_y, '(b)', transform=ax[0,1].transAxes)
ax[1,1].text(fig_label_x, fig_label_y, '(c)', transform=ax[1,1].transAxes)
ax[2,1].text(fig_label_x, fig_label_y, '(d)', transform=ax[2,1].transAxes)
for i in range(0,3):
 
    ax[i,1].set_xlabel('')
    ax[i,1].set_ylabel('')


    ax[i,1].set_xlim([0.05,0.95])
    ax[i,1].set_ylim([0.3,1])
   
ax[2,1].set_xlabel('q')
ax[1,1].set_ylabel(r'$\rho$',rotation='horizontal')

plt.tight_layout(h_pad=0.1)
plt.show()

In [ ]:
baseSavePath=''
saveFigPath = os.path.join(baseSavePath,'Figure2.pdf')
fig.savefig(saveFigPath,bbox_inches='tight', pad_inches=0.05)